# AC Stark Shift (Fig. 2e,f)

This notebook rebuilds the AC Stark shift panels shown in the attached figure:

- **Panel e:** frequency sweep of qubit/antiqubit Stark shifts with dispersive-model fits.
- **Panel f:** amplitude sweep at fixed drive, replotted versus calibrated \(\Omega_s/(2\pi)\).

**Inputs**
- Hard-coded frequency-sweep arrays from `notebooks/data_analysis.ipynb`.
- `data/ACS_amp_sweep_Amp_1213_1035.csv`
- `data/ACS_amp_sweep_Q1_delta_freq_amp_array_1213_1035.csv`
- `data/ACS_amp_sweep_Q2_delta_freq_amp_array_1213_1035.csv`

**Outputs**
- `main_text_figures/fig2ef_ac_stark_shift.png`
- `main_text_figures/fig2ef_ac_stark_shift.pdf`


In [ ]:
import os
from pathlib import Path

# Ensure relative paths work regardless of where the notebook is launched from.
_repo_root = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").exists():
        _repo_root = p
        break
if _repo_root is not None:
    os.chdir(_repo_root)
print("Working directory:", Path.cwd())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Global style tuned to match panel look in the target figure.
lw = 1.0
mpl.rcParams['axes.linewidth'] = lw
mpl.rcParams['lines.linewidth'] = lw
mpl.rcParams['xtick.major.width'] = lw
mpl.rcParams['ytick.major.width'] = lw
mpl.rcParams['xtick.major.size'] = 3
mpl.rcParams['ytick.major.size'] = 3
mpl.rcParams['font.size'] = 8
mpl.rcParams['axes.labelsize'] = 8
mpl.rcParams['legend.fontsize'] = 8
mpl.rcParams['xtick.labelsize'] = 8
mpl.rcParams['ytick.labelsize'] = 8
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['mathtext.rm'] = 'STIXGeneral'
mpl.rcParams['mathtext.it'] = 'STIXGeneral:italic'
mpl.rcParams['mathtext.bf'] = 'STIXGeneral:bold'


## Load Data


In [ ]:
# Frequency-sweep data copied from notebooks/data_analysis.ipynb.
acs_frequency_ghz = np.linspace(4.10, 4.35, num=26)

# q (Q1) and qbar (Q2) AC Stark shifts in Hz.
delta_q_hz = np.array([
    456358, 440364, 479197, 457785, 382023, 405096, 817702, -2190954,
    -750987, -511354, -427594, -345728, -247638, -183449, -145183,
    -98772, -54644, -35016, -30891, -38599, -40378, -40025, -38624,
    -36081, -29740, -24528
], dtype=float)

delta_qbar_hz = np.array([
    -596512, -756523, -1360695, 2114717, 1010825, 671982, 727803, 903658,
    898561, 925287, 1064361, 1004260, 901459, 857606, 967765, 1163793,
    1511051, 3548153, -2950243, -1592068, -1038679, -704298, -502514,
    -389556, -295553, -214057
], dtype=float)

# Keep the same x-window as panel e in the target image.
panel_e_mask = acs_frequency_ghz <= 4.30

# Amplitude-sweep data from fig2f.ipynb/data folder.
data_dir = Path("data")
file_amp = data_dir / "ACS_amp_sweep_Amp_1213_1035.csv"
file_q_amp = data_dir / "ACS_amp_sweep_Q1_delta_freq_amp_array_1213_1035.csv"
file_qbar_amp = data_dir / "ACS_amp_sweep_Q2_delta_freq_amp_array_1213_1035.csv"

for fp in [file_amp, file_q_amp, file_qbar_amp]:
    if not fp.exists():
        raise FileNotFoundError(f"Missing input file: {fp}")

amp_arb = pd.read_csv(file_amp)["amp"].to_numpy(dtype=float)
delta_q_amp_hz = pd.read_csv(file_q_amp)["Q1_delta_freq"].to_numpy(dtype=float)
delta_qbar_amp_hz = pd.read_csv(file_qbar_amp)["Q2_delta_freq"].to_numpy(dtype=float)

# Empirical calibration used in the original analysis flow: Ωs/(2π) = 16 MHz per amplitude unit.
omega_over_2pi_mhz_per_amp = 16.0
omega_s_over_2pi_mhz = omega_over_2pi_mhz_per_amp * amp_arb

print("Frequency-sweep points:", panel_e_mask.sum())
print("Amplitude-sweep points:", len(amp_arb))


## Fit Dispersive Model For Panel e

This notebook now uses the same fitting algorithm as `notebooks/data_analysis.ipynb`:

- model:
  \[
  \delta(\omega_d)=\frac{\alpha\,\text{scale}}{2\,\Delta\,(\alpha+\Delta)},\quad \Delta=\omega_q-\omega_d
  \]
- nonlinear least-squares (`scipy.optimize.curve_fit`) fitting only the scale factor for each qubit
- `\bar{q}` (`Q2`) is then refit on the filtered range `ACS_freq > 4.13e9`, exactly as in `data_analysis.ipynb`.


In [ ]:
from scipy.optimize import curve_fit

# Standalone replacements for the device constants used via cf.* in data_analysis.ipynb.
# These values keep this notebook runnable without the lab-specific config module.
Q1_FREQ_HZ = 4.167131661442007e9
Q1_EF_FREQ_HZ = Q1_FREQ_HZ - 84.23197492163008e6
Q2_FREQ_HZ = 4.275442110405471e9
Q2_EF_FREQ_HZ = Q2_FREQ_HZ - 154.86240026054388e6


def delta_Q_freq(ACS_freq, Q_freq, Q_ef_freq, scale_factor):
    # Same function form as notebooks/data_analysis.ipynb
    alpha = Q_ef_freq - Q_freq
    delta_qs = Q_freq - ACS_freq
    delta_Q_freq = alpha * scale_factor / (2 * delta_qs * (alpha + delta_qs))
    return delta_Q_freq


def mask_extreme_values(y_mhz, limit_mhz=3.4):
    y = np.array(y_mhz, dtype=float, copy=True)
    y[np.abs(y) > limit_mhz] = np.nan
    return y


drive_panel_e_hz = acs_frequency_ghz[panel_e_mask] * 1e9
q_panel_e_hz = delta_q_hz[panel_e_mask]
qbar_panel_e_hz = delta_qbar_hz[panel_e_mask]

# Initial guesses follow data_analysis.ipynb.
scale_factor_Q1 = 1e13
scale_factor_Q2 = 1.5e13

# Q1 fit
popt_Q1, pcov_Q1 = curve_fit(
    lambda ACS_freq, scale_factor: delta_Q_freq(ACS_freq, Q1_FREQ_HZ, Q1_EF_FREQ_HZ, scale_factor),
    drive_panel_e_hz,
    q_panel_e_hz,
    p0=[scale_factor_Q1],
)
scale_factor_Q1 = popt_Q1[0]

# Q2 fit on full range
popt_Q2, pcov_Q2 = curve_fit(
    lambda ACS_freq, scale_factor: delta_Q_freq(ACS_freq, Q2_FREQ_HZ, Q2_EF_FREQ_HZ, scale_factor),
    drive_panel_e_hz,
    qbar_panel_e_hz,
    p0=[scale_factor_Q2],
)
scale_factor_Q2 = popt_Q2[0]

# Q2 filtered refit (exactly as done in data_analysis.ipynb)
filter_condition = drive_panel_e_hz > 4.13e9
drive_panel_e_filtered_hz = drive_panel_e_hz[filter_condition]
qbar_panel_e_filtered_hz = qbar_panel_e_hz[filter_condition]

popt_Q2_filtered, pcov_Q2_filtered = curve_fit(
    lambda ACS_freq, scale_factor: delta_Q_freq(ACS_freq, Q2_FREQ_HZ, Q2_EF_FREQ_HZ, scale_factor),
    drive_panel_e_filtered_hz,
    qbar_panel_e_filtered_hz,
    p0=[scale_factor_Q2],
)
scale_factor_Q2 = popt_Q2_filtered[0]

# Smooth curves for plotting
# Keep panel-e curves in the same x-window as the target panel.
drive_dense_ghz = np.linspace(4.10, 4.30, 3000)
drive_dense_hz = drive_dense_ghz * 1e9

q_fit_dense_mhz = delta_Q_freq(
    drive_dense_hz,
    Q1_FREQ_HZ,
    Q1_EF_FREQ_HZ,
    scale_factor_Q1,
) * 1e-6

qbar_fit_dense_mhz = delta_Q_freq(
    drive_dense_hz,
    Q2_FREQ_HZ,
    Q2_EF_FREQ_HZ,
    scale_factor_Q2,
) * 1e-6

# Break curve lines near poles so the rendering does not connect through asymptotes.
q_fit_line_mhz = mask_extreme_values(q_fit_dense_mhz, limit_mhz=3.4)
qbar_fit_line_mhz = mask_extreme_values(qbar_fit_dense_mhz, limit_mhz=3.4)

# Quadratic fits for panel-f trend lines.
q_amp_mhz = delta_q_amp_hz * 1e-6
qbar_amp_mhz = delta_qbar_amp_hz * 1e-6
q_amp_poly = np.polyfit(omega_s_over_2pi_mhz, q_amp_mhz, deg=2)
qbar_amp_poly = np.polyfit(omega_s_over_2pi_mhz, qbar_amp_mhz, deg=2)
omega_dense = np.linspace(0, max(omega_s_over_2pi_mhz) + 0.2, 500)



## Plot Panels e and f


In [ ]:
green = '#119911'
blue = '#1E33D9'
panel_bg = 'white'

fig_size = np.asarray([8.6, 12.0]) / 2.54
fig = plt.figure(figsize=fig_size, dpi=400)
fig.patch.set_facecolor('white')
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.0], hspace=0.35)

# ----------------------------- Panel e -----------------------------
ax_e = fig.add_subplot(gs[0, 0])
ax_e.set_facecolor(panel_bg)

ax_e.plot(drive_dense_ghz, qbar_fit_line_mhz, '--', color=green, linewidth=1.2)
ax_e.plot(drive_dense_ghz, q_fit_line_mhz, '--', color=blue, linewidth=1.2)

ax_e.scatter(
    acs_frequency_ghz[panel_e_mask],
    qbar_panel_e_hz * 1e-6,
    s=18,
    marker='s',
    color=green,
    label=r'$\bar{q}$',
    zorder=3,
)
ax_e.scatter(
    acs_frequency_ghz[panel_e_mask],
    q_panel_e_hz * 1e-6,
    s=20,
    marker='o',
    color=blue,
    label=r'$q$',
    zorder=3,
)

# Pole markers.
omega_q_ghz = Q1_FREQ_HZ * 1e-9
omega_qbar_ghz = Q2_FREQ_HZ * 1e-9
ax_e.axvline(omega_q_ghz, color=blue, linestyle=':', linewidth=1.0)
ax_e.axvline(omega_qbar_ghz, color=green, linestyle=':', linewidth=1.0)

# Requested reference line at fixed drive frequency.
ax_e.axvline(4.177, color='gray', linestyle='-', linewidth=1.0)

ax_e.text(
    omega_q_ghz,
    1.03,
    r'$\omega_q/(2\pi)$',
    ha='center',
    va='bottom',
    transform=ax_e.get_xaxis_transform(),
)
ax_e.text(
    omega_qbar_ghz,
    1.03,
    r'$\omega_{\bar{q}}/(2\pi)$',
    ha='center',
    va='bottom',
    transform=ax_e.get_xaxis_transform(),
)

ax_e.set_xlim(4.095, 4.30)
ax_e.set_ylim(-2.6, 2.6)
ax_e.set_xticks([4.10, 4.20, 4.30])
ax_e.set_yticks([-2, 0, 2])
ax_e.tick_params(direction='in', top=True, right=True)
ax_e.set_xlabel('Drive freq. (GHz)')
ax_e.set_ylabel(r'$\delta_q,\delta_{\bar q}/(2\pi)$ (MHz)')
ax_e.legend(loc='lower right', frameon=False, handlelength=1.0, handletextpad=0.3)
ax_e.text(-0.28, 1.03, 'e', transform=ax_e.transAxes, fontsize=13, fontweight='bold')

# ----------------------------- Panel f -----------------------------
ax_f = fig.add_subplot(gs[1, 0])
ax_f.set_facecolor(panel_bg)

ax_f.plot(omega_dense, np.polyval(qbar_amp_poly, omega_dense), color=green, linewidth=1.2)
ax_f.plot(omega_dense, np.polyval(q_amp_poly, omega_dense), color=blue, linewidth=1.2)

ax_f.scatter(
    omega_s_over_2pi_mhz,
    qbar_amp_mhz,
    s=18,
    marker='s',
    color=green,
    label=r'$\bar{q}$',
    zorder=3,
)
ax_f.scatter(
    omega_s_over_2pi_mhz,
    q_amp_mhz,
    s=20,
    marker='o',
    color=blue,
    label=r'$q$',
    zorder=3,
)

ax_f.set_xlim(-0.3, 7.5)
ax_f.set_ylim(-3.3, 3.1)
ax_f.set_xticks([0, 2, 4, 6])
ax_f.set_yticks([-2, 0, 2])
ax_f.tick_params(direction='in', top=True, right=True)
ax_f.set_xlabel(r'$\Omega_s/(2\pi)$ (MHz)')
ax_f.set_ylabel(r'$\delta_q,\delta_{\bar q}/(2\pi)$ (MHz)')
ax_f.legend(loc='upper left', frameon=False, handlelength=2.0, handletextpad=0.4)
ax_f.text(-0.28, 1.03, 'f', transform=ax_f.transAxes, fontsize=13, fontweight='bold')

# Save outputs.
out_png = Path('main_text_figures') / 'fig2ef_ac_stark_shift.png'
out_pdf = Path('main_text_figures') / 'fig2ef_ac_stark_shift.pdf'
out_png.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(out_png, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Saved: {out_png}")
print(f"Saved: {out_pdf}")
print()
print("Panel-e fit summary")
print(f"Q1 scale_factor / 1e13 = {scale_factor_Q1 / 1e13:.6f}")
print(f"Q2 scale_factor / 1e13 = {scale_factor_Q2 / 1e13:.6f}")
print(f"Q1 omega_q/(2pi) = {omega_q_ghz:.6f} GHz")
print(f"Q2 omega_q/(2pi) = {omega_qbar_ghz:.6f} GHz")

